#Cours 4 - Job et orchestration
Ce notebook contient un code simple qui va récupérer un fichier à partir d'un volume et le charger dans une table orchestration_test.

### Les etapes du job
- Vérification de la présence d'un fichier JSON dans le volume : /Volumes/adbi_training/training/training/orchestration/Cours4/data_in
- Vérification de l'existence de la table
- Parsing du fichier JSON
- Insertion dans la table


## Etape 1 :
Vérification de la présence d'un fichier JSON dans le volume : /Volumes/workspace/databricks_training/training/orchestration/Cours4/data_in/

In [0]:
# vérification de la présence d'un fichier JSON dans le dossier
import os

volume_path = dbutils.widgets.get("volume_path")
#volume_path = '/Volumes/adbi_training/training/training/orchestration/Cours4/data_in/'

base_path = os.path.dirname(volume_path.rstrip('/'))

files = dbutils.fs.ls(volume_path)
json_files = [f.path for f in files if f.path.lower().endswith('.json')]

if json_files:
    print("JSON files found:", json_files)
else:
    dbutils.notebook.exit("No JSON files found in the volume.")

## Etape 2 : 

Vérification de l'existence de la table

In [0]:
# Vérification de l'existence de la table orchestration_test
spark.sql("USE CATALOG adbi_training")
spark.sql("USE SCHEMA training")
table_exists = spark.catalog.tableExists("orchestration_test")
print(f"La table orchestration_test existe : {table_exists}")

if (table_exists == False):
  print(f"Création de la table orchestration_test")
  # Création de la table orchestration_test dans le catalog adbi_training et le schema training si elle n'existe pas déjà
  spark.sql("""
  CREATE TABLE IF NOT EXISTS adbi_training.training.orchestration_test (
    ID INTEGER,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    EMAIL STRING
  ) USING DELTA
  """)
  print(f"La table orchestration_test existe : {spark.catalog.tableExists("orchestration_test")}")

### Etape 3 et 4 : 
Parsing du fichier JSON dans un Dataframe Spark et insertion en table


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from datetime import datetime

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True)
])

# récupération nombre de lignes dans la table
nb_lignes = spark.sql("SELECT COUNT(*) FROM adbi_training.training.orchestration_test").collect()

for json_path in json_files:
  print(f"Traitement du fichier {json_path}")
  df = spark.read.schema(schema).json(json_path)
  df = df.dropna()
  df.writeTo("adbi_training.training.orchestration_test").append()
  
  # vérification de la présence des données dans la table orchestration_test
  nb_lignes_after_insert = spark.sql("SELECT COUNT(*) FROM adbi_training.training.orchestration_test").collect()
  if nb_lignes_after_insert > nb_lignes:
    print(f"Données insérées dans la table orchestration_test")
    # Déplacer le fichier JSON traité vers un autre dossier
    destination_folder = f"{base_path}/processed/{datetime.now()}/"
    dbutils.fs.mkdirs(destination_folder)
    destination_path = destination_folder + json_path.split("/")[-1]
    dbutils.fs.mv(json_path, destination_path)
    dbutils.fs.rm(json_path)
  else:
    # Déplacer le fichier JSON traité vers un autre dossier
    destination_folder = f"{base_path}/rejects/{datetime.now()}/"
    dbutils.fs.mkdirs(destination_folder)
    destination_path = destination_folder + json_path.split("/")[-1]
    dbutils.fs.mv(json_path, destination_path)
    dbutils.fs.rm(json_path)
    raise Exception("Aucune donnée insérée dans la table orchestration_test")
    
